# 03 Gamma Forecast Impact

This notebook quantifies how the selected Gamma RPF case can inflate 7-day-ahead forecast error.

**Inputs.** It reads `dataset/final/dataset_gamma.parquet` from Notebook 0 and, in full mode, uses Alpha training data for the `m8_xgb` correction model.

**Outputs.** It writes Gamma forecast examples, forecast metrics, a perfect-model baseline table, and paper-facing figures under `outputs/*/03_gamma_forecast_impact/`.

**Key decisions.** Fig01 shows 2024-09-01 to 2024-09-07, the start of the September 2024 forecast test period. Forecast labels use `Uncorrected data`, `m8_xgb-corrected data`, and `Manually corrected data` so the manuscript language is consistent.

**Run modes.** Smoke mode writes deterministic placeholder forecast-model rows for layout checks without training linear regression or XGBoost. The perfect-model baseline rows are deterministic comparisons and remain real in smoke mode.


## 1. Imports And Paths

Load config, resolve output folders, and confirm whether the notebook is in smoke or full forecast mode. The shared helper keeps the forecast construction and plotting logic in one place for easier testing.


In [ ]:
from pathlib import Path
import sys

# Keep notebook imports stable whether the notebook is run from JupyterLab,
# VS Code, or the repository root.
article_root = Path.cwd()
while article_root.name != "2_journal_article":
    if article_root.parent == article_root:
        raise RuntimeError("Could not locate publication/2_journal_article")
    article_root = article_root.parent
notebook_dir = article_root / "notebooks"
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

import _experiment_helpers as h

cfg = h.load_config(article_root)
paths = h.article_paths(article_root, cfg)
h.ensure_output_dirs(paths)
print(f"Article root: {article_root}")
print(f"Config schema: {cfg['schema_version']}")
print(f"Output root: {paths.outputs}")

print(article_root / cfg["paths"]["gamma_dataset_path"])
print(f"Full forecast: {cfg['execution']['run_full_forecast']}")


## 2. Load Gamma And Confirm Scope

`h.load_dataset()` validates that Gamma contains exactly one selected Beta site over the same one-year window as Beta. This matters because the forecast-impact story is a focused case study rather than a population-wide forecast experiment.


In [ ]:
# Load Gamma before running the workflow so the selected site and row counts are visible.
gamma = h.load_dataset(article_root, cfg, "gamma")
site = gamma["substation_id"].iloc[0]
print(f"Gamma site: {site}")
h.dataset_summary(gamma, "Gamma")


## 3. Run Forecast-Impact Workflow

`h.run_gamma_forecast_impact()` prepares the uncorrected, m8-corrected, and manually corrected series; writes the perfect-model baseline; builds 7-day-ahead forecast examples; creates smoke or full forecast rows; computes RMSE/MAE; writes tables, figures, and the manifest. In full mode, each target uses only history available up to seven days before the target timestamp.


In [ ]:
# Smoke mode previews output shape; full mode trains correction and forecast models.
result = h.run_gamma_forecast_impact(article_root)
print(result["status"], result["gamma_site"])
result["metrics"]


## 4. Interpret The Perfect-Model Baseline

The `perfect_model_baseline` rows answer a narrow but powerful question: if a model perfectly predicted each data condition, how much RMSE remains against manually corrected data? The uncorrected-data row isolates RPF sign-error impact; the m8_xgb-corrected row reflects remaining correction error; the manually corrected row should be zero. Forecast-model rows add model error on top of those data-condition effects.


In [ ]:
result["metrics"].sort_values(["data_condition", "model"])
